# 08 - Generation-response evaluation analysis

We analyse the 140 blinded judgments against the frozen 20-question, seven-condition design. The judgments were validated and imported before this stage. Our scores are final for analysis, while the professor's current feedback remains preliminary; later comments on the overall system can inform the thesis discussion without changing these scores. We preserve the exact judgment fingerprint and the separate private status records.

We only display and export aggregate counts, accuracy, secondary-score distributions, paired comparisons and descriptive strata. Candidate text, reference solutions, judgment notes, question identifiers and answer-level scores remain private. Notebook 07 records reference validation; the private grouped notebook holds the original scoring record.

In [1]:
from __future__ import annotations

import hashlib
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "configs/generation-response-evaluation-config.json").is_file():
    raise FileNotFoundError("The thesis repository root was not found.")

sys.path.insert(0, str(PROJECT_ROOT / "src"))
from geotech_rag.response_analysis import (
    compute_aggregates, load_frozen_analysis, output_payload, save_public_aggregates,
)

RUN_WRITE_PUBLIC_RESULTS = False
print("Repository:", PROJECT_ROOT)
print("Write public aggregate results:", RUN_WRITE_PUBLIC_RESULTS)

Repository: /home/zaki/coding_F/GismaProjects/Thesis
Write public aggregate results: False


## 1. Recheck the frozen lineage

We verify the reference, question, generated response, completed review notebook, imported judgments and both private decision records against their saved fingerprints. We match all 140 blinded IDs to their frozen question-condition pairs only in memory. The score file contains neither condition identities nor response text. These checks stop analysis if one input has changed or a question-condition pair is missing.

The earlier private status record reports preliminary professor feedback. A later private decision records our instruction that the 140 scores are final for analysis. Neither record replaces the expert judgment file.

In [2]:
config, questions, references, paired_rows, fingerprints = load_frozen_analysis(PROJECT_ROOT)
print("Verified frozen questions:", len(questions))
print("Verified paired judgments:", len(paired_rows))
print("Judgment SHA-256:", fingerprints["judgments_sha256"])
print("Score-set decision SHA-256:", fingerprints["final_score_set_decision_sha256"])
print("Question and answer text displayed: False")

Verified frozen questions: 20
Verified paired judgments: 140
Judgment SHA-256: ad861d0d40bc4bfa59a0bda5b6f57abbff0aaac4b772cb9daad7adc4317ca567
Score-set decision SHA-256: 1501da1eef6c850604d7e75c2d1da2faf3738b5279f7c39768eff06111cd12ad
Question and answer text displayed: False


## 2. Report answer correctness and completion

We use binary correctness as the primary outcome, with all 20 prevalidated questions in every condition. Accuracy is the number correct divided by 20. We report Wilson 95% confidence intervals and visible-response completion separately. The nine frozen responses without visible text stay in the accuracy denominator and score incorrect. We do not infer within-condition sampling variance from a single generated answer per question.

In [3]:
aggregate = compute_aggregates(questions, references, paired_rows, fingerprints)
display(pd.DataFrame(aggregate["condition_summary"]))

,condition,gradable_questions,correct,accuracy,wilson95_lower,wilson95_upper,visible_responses,completion_rate,no_visible_answers,unsupported_claims_count
0,P0,20,2,0.10,0.027866,0.301034,20,1.00,0,15
1,P1,20,3,0.15,0.052369,0.360419,20,1.00,0,14
2,P2,20,3,0.15,0.052369,0.360419,20,1.00,0,15
3,P3,20,2,0.10,0.027866,0.301034,20,1.00,0,18
4,M1,20,10,0.50,0.299298,0.700702,20,1.00,0,5
5,M2,20,11,0.55,0.342085,0.741802,11,0.55,9,0
6,M3,20,20,1.00,0.838875,1.000000,20,1.00,0,0


## 3. Compare matched conditions

Every condition answered the same 20 questions. We compare P0 with P1 using exact McNemar; for P1/P2/P3 and P1/M1/M2/M3 we use Cochran's Q, then exact pairwise McNemar comparisons. Holm adjustment applies within each planned family. Tests with no discordance have p = 1. These are paired analyses; an unpaired chi-square calculation cannot replace them.

The original paper's GPT-4/Llama-3 pair is absent from our seven frozen conditions. We therefore do not report its unpaired chi-square comparison as though it were reproduced.

In [4]:
display(pd.DataFrame.from_dict(aggregate["binary_omnibus"], orient="index").reset_index(names="family"))
display(pd.DataFrame(aggregate["binary_pairwise"]))

,family,statistic,df,p_value,no_discordance
0,temperature,0.666667,2,7.165313e-01,False
1,model_generation,31.285714,3,7.400764e-07,False


,family,left_condition,right_condition,left_only_correct,right_only_correct,discordant,p_value,holm_adjusted_p_value
0,retrieval,P0,P1,0,1,1,1.000000,1.000000
1,temperature,P1,P2,1,1,2,1.000000,1.000000
2,temperature,P1,P3,1,0,1,1.000000,1.000000
3,temperature,P2,P3,2,1,3,1.000000,1.000000
4,model_generation,P1,M1,0,7,7,0.015625,0.031250
5,model_generation,P1,M2,0,8,8,0.007812,0.023438
6,model_generation,P1,M3,0,17,17,0.000015,0.000092
7,model_generation,M1,M2,2,3,5,1.000000,1.000000
8,model_generation,M1,M3,0,10,10,0.001953,0.009766
9,model_generation,M2,M3,0,9,9,0.003906,0.015625


## 4. Describe secondary judgments and errors

We summarise formula integration, calculation correctness, unit correctness, explanation clarity and task adaptability on their frozen 0-2 scales. The first three use only questions marked applicable in the approved reference; the same eligible questions enter every compared condition. For two conditions we use paired Wilcoxon signed-rank tests, and for three or four conditions we use Friedman tests followed by paired Wilcoxon comparisons with Holm adjustment. Tied scores and pairs without a difference remain accounted for.

We report one primary error category per incorrect response, including the documented `no_visible_answer` extension. Problem-type and cognitive-level summaries are descriptive: ten reference records lack these classifications and are labelled `not_recorded_in_reference`, rather than being assigned a category from generated answers. Recorded categories with fewer than three questions are pooled before we publish results. All recorded problem types are pooled here; small strata are interpreted descriptively. We also count unsupported claims per condition.

In [5]:
display(pd.DataFrame(aggregate["secondary_summary"]))
display(pd.DataFrame(aggregate["error_counts"]))
for family, comparisons in aggregate["ordinal_comparisons"].items():
    print("Ordinal comparison family:", family)
    for field, result in comparisons.items():
        print(field, "common applicable questions:", result["common_applicable_questions"],
              "Friedman:", result["friedman"])
        display(pd.DataFrame(result["paired_wilcoxon"]))
for field, records in aggregate["adaptability_strata"].items():
    print("Descriptive stratum:", field)
    display(pd.DataFrame(records))

,condition,field,applicable_questions,mean_score,median_score,score_0_count,score_1_count,score_2_count
0,P0,formula_integration,17,1.000000,1.0,3,11,3
1,P1,formula_integration,17,1.235294,1.0,1,11,5
2,P2,formula_integration,17,1.235294,1.0,2,9,6
3,P3,formula_integration,17,1.000000,1.0,3,11,3
4,M1,formula_integration,17,1.764706,2.0,0,4,13
5,M2,formula_integration,17,1.294118,2.0,6,0,11
6,M3,formula_integration,17,2.000000,2.0,0,0,17
7,P0,calculation_correctness,20,0.200000,0.0,18,0,2
8,P1,calculation_correctness,20,0.400000,0.0,15,2,3
9,P2,calculation_correctness,20,0.450000,0.0,15,1,4


,condition,primary_error_category,count,denominator
0,P0,grounding,7,20
1,P0,conceptual,6,20
2,P0,calculation,3,20
3,P0,deficiency,2,20
4,P0,no_visible_answer,0,20
5,P0,other,0,20
6,P1,grounding,5,20
7,P1,conceptual,7,20
8,P1,calculation,2,20
9,P1,deficiency,3,20


Ordinal comparison family: retrieval
formula_integration common applicable questions: 17 Friedman: None


,left_condition,right_condition,nonzero_pairs,statistic,p_value,holm_adjusted_p_value
0,P0,P1,5,2.5,0.3125,0.3125


calculation_correctness common applicable questions: 20 Friedman: None


...

,left_condition,right_condition,nonzero_pairs,statistic,p_value,holm_adjusted_p_value
0,P1,M1,7,0.0,0.015625,0.046875
1,P1,M2,16,56.0,0.650879,0.650879
2,P1,M3,15,0.0,0.000061,0.000366
3,M1,M2,10,4.5,0.017578,0.046875
4,M1,M3,8,0.0,0.007812,0.031250
5,M2,M3,9,0.0,0.003906,0.019531


Descriptive stratum: problem_type


,stratum,condition,question_count,correct,mean_adaptability
0,not_recorded_in_reference,P0,10,2,1.2
1,not_recorded_in_reference,P1,10,2,1.3
2,not_recorded_in_reference,P2,10,2,1.4
3,not_recorded_in_reference,P3,10,1,1.2
4,not_recorded_in_reference,M1,10,4,1.6
5,not_recorded_in_reference,M2,10,6,1.2
6,not_recorded_in_reference,M3,10,10,2.0
7,recorded_not_stratified,P0,10,0,1.1
8,recorded_not_stratified,P1,10,1,1.2
9,recorded_not_stratified,P2,10,1,1.1


Descriptive stratum: cognitive_level


,stratum,condition,question_count,correct,mean_adaptability
0,Apply,P0,10,0,1.1
1,Apply,P1,10,1,1.2
2,Apply,P2,10,1,1.1
3,Apply,P3,10,1,1.1
4,Apply,M1,10,6,1.6
5,Apply,M2,10,5,1.0
6,Apply,M3,10,10,2.0
7,not_recorded_in_reference,P0,10,2,1.2
8,not_recorded_in_reference,P1,10,2,1.3
9,not_recorded_in_reference,P2,10,2,1.4


## 5. Save only aggregate outputs

We first preview file paths, byte counts and fingerprints. The write flag remains disabled until the public-only results are checked. The five outputs contain the condition summary, applicable secondary scores, paired binary comparisons, error distributions and the full aggregate metric record. We create each output only once and refuse to replace an existing result silently. No private question, answer, note, response ID or per-question judgment is exported.

In [6]:
payload = output_payload(PROJECT_ROOT, config, aggregate)
for path, content in payload.items():
    print(path.relative_to(PROJECT_ROOT), len(content), hashlib.sha256(content).hexdigest())
if RUN_WRITE_PUBLIC_RESULTS:
    save_public_aggregates(payload)
    print("Five aggregate outputs saved without overwrite.")
else:
    print("Public write disabled; preview only.")

results/tables/generation-evaluation.csv 588 8864bda3284f3addc0c849ae4ed56286a685f5635f0f47a615535fdceb3acd2c
results/tables/generation-secondary-scores.csv 1672 aaccfb89ca8f1173a959a0b1b73885749fff6d6acd7da1e9e31bc01c2efb013c
results/tables/generation-pairwise-comparisons.csv 544 3b3fa163971e68b4d4fa1400aa3a1ab33c9b7a8baa123612968a9895bafa3c5a
results/tables/generation-error-distribution.csv 863 dacfbfb0b7c930b113c164507e26e347775af13b6e2e35f9a7754c92cb4dae19
results/metrics/generation-evaluation.json 40908 134d28d43ae605b2eb06659b2bd4a71a9342a2cdd856fd9ad7cd5f1ed97dc253
Public write disabled; preview only.


## 6. Interpretation boundaries

We compare conditions under one frozen paired benchmark. We will discuss grouped presentation, one expert, the nine no-visible-answer responses and preliminary professor feedback when interpreting the results. We report the professor's later overall-system comment separately from answer scores. The baseline paper reports 82.5% in one setting, but the unpublished calculation behind that value cannot be reconstructed as binary correctness over exactly 20 questions; our score remains an explicit correct/20 measure. Any later correction to an individual score would require the documented private correction and analysis to be rerun before claims are final.